In [ ]:
#TD learning
import random

V = {}
ALPHA = 0.1
EPSILON = 0.2
EPISODES = 3

def state(board):
    return ''.join(board)

def empty(board):
    return [i for i in range(9) if board[i] == ' ']

def winner(board, p):
    wins = [(0,1,2), (3,4,5), (6,7,8),
            (0,3,6), (1,4,7), (2,5,8),
            (0,4,8), (2,4,6)]
    return any(all(board[i] == p for i in w) for w in wins)

def value(s):
    if s not in V:
        V[s] = 0.5
    return V[s]

def choose_move(board):
    moves = empty(board)

    if random.random() < EPSILON:
        return random.choice(moves)

    best = moves[0]
    bestv = -1

    for m in moves:
        b = board.copy()
        b[m] = 'X'
        v = value(state(b))

        if v > bestv:
            bestv = v
            best = m

    return best

# Training
wins = draws = losses = 0

for ep in range(EPISODES):
    board = [' '] * 9
    visited = []

    while True:
        # Agent move
        a = choose_move(board)
        board[a] = 'X'
        st = state(board)
        if st not in V:
          V[st] = 0.5
        visited.append(st)


        if winner(board, 'X'):
            reward = 1.0
            wins += 1
            break

        if not empty(board):
            reward = 0.5
            draws += 1
            break

        # Random opponent move
        o = random.choice(empty(board))
        board[o] = 'O'
        st = state(board)
        if st not in V:
          V[st] = 0.5
        visited.append(st)

        if winner(board, 'O'):
            reward = 0.0
            losses += 1
            break

        if not empty(board):
            reward = 0.5
            draws += 1
            break

    # TD-style backward update
    target = reward

    for s in reversed(visited):
        V[s] = V[s] + ALPHA * (target - V[s])
        target = V[s]

print("Training completed")
print("Wins:", wins)
print("Draws:", draws)
print("Losses:", losses)
print("States learned:", len(V))

Training completed
Wins: 3
Draws: 0
Losses: 0
States learned: 47


In [ ]:
#TD with dynamic update
import random

# Parameters
ROWS, COLS = 4, 4
START = (0, 0)
GOAL = (3, 3)
RESTRICTED = [(1, 1), (2, 2)]

ALPHA = 0.1
EPISODES = 5
MAX_STEPS = 500

ACTIONS = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1)
}

# Initialize state values
V = {}

for r in range(ROWS):
    for c in range(COLS):
        if (r, c) not in RESTRICTED:
            V[(r, c)] = 0.0

# Move agent
def move_agent(state, action):
    r, c = state
    dr, dc = ACTIONS[action]

    next_state = (r + dr, c + dc)

    if not (0 <= next_state[0] < ROWS and
            0 <= next_state[1] < COLS):
        return state

    if next_state in RESTRICTED:
        return state

    return next_state

# TD(0) Training
successes = 0

for ep in range(EPISODES):
    state = START

    for step in range(MAX_STEPS):
        action = random.choice(list(ACTIONS.keys()))
        next_state = move_agent(state, action)

        reward = 1 if next_state == GOAL else 0

        # TD(0) update
        td_error = reward + V[next_state] - V[state]
        V[state] += ALPHA * td_error

        state = next_state

        if state == GOAL:
            successes += 1
            break

# Final output
print("Training Completed")
print("Episodes reaching goal:", successes)

print("\nFinal State Values:")
for r in range(ROWS):
    for c in range(COLS):
        s = (r, c)

        if s in RESTRICTED:
            print("   X   ", end=" ")
        else:
            print(f"{V[s]:.3f}", end=" ")
    print()


Training Completed
Episodes reaching goal: 5

Final State Values:
0.000 0.000 0.000 0.000 
0.002    X    0.000 0.016 
0.010 0.036    X    0.084 
0.032 0.066 0.253 0.000 


In [ ]:
#Q-learning
import random

# Parameters
ROWS, COLS = 4, 4
START = (0, 0)
GOAL = (3, 3)
RESTRICTED = [(1, 1), (2, 2)]

ALPHA = 0.1
EPSILON = 0.1
EPISODES = 1000
MAX_STEPS = 500

ACTIONS = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1)
}

ACTION_NAMES = list(ACTIONS.keys())

# Initialize Q-values
Q = {}

for r in range(ROWS):
    for c in range(COLS):
        for action in ACTION_NAMES:
            Q[((r, c), action)] = 0.0

# Move agent
def move_agent(state, action):
    r, c = state
    dr, dc = ACTIONS[action]

    next_state = (r + dr, c + dc)

    if not (0 <= next_state[0] < ROWS and
            0 <= next_state[1] < COLS):
        return state

    if next_state in RESTRICTED:
        return state

    return next_state

# Epsilon-greedy action selection
def choose_action(state):
    if random.random() < EPSILON:
        return random.choice(ACTION_NAMES)

    max_q = max(Q[(state, a)] for a in ACTION_NAMES)

    best_actions = [
        a for a in ACTION_NAMES
        if Q[(state, a)] == max_q
    ]

    return random.choice(best_actions)

# Q-learning training
successes = 0

for ep in range(EPISODES):
    state = START

    for step in range(MAX_STEPS):
        action = choose_action(state)
        next_state = move_agent(state, action)

        reward = 1 if next_state == GOAL else 0

        old_q = Q[(state, action)]

        if next_state == GOAL:
            next_q = 0
        else:
            next_q = max(
                Q[(next_state, a)]
                for a in ACTION_NAMES
            )

        td_error = reward + next_q - old_q
        Q[(state, action)] += ALPHA * td_error

        state = next_state

        if state == GOAL:
            successes += 1
            break

print("Training Completed")
print("Episodes reaching goal:", successes)

print("\nBest Action for Each State:")

for r in range(ROWS):
    for c in range(COLS):
        state = (r, c)

        if state == GOAL:
            print(f"{'GOAL':^8}", end=" ")
        elif state in RESTRICTED:
            print(f"{'X':^8}", end=" ")
        else:
            best = max(
                ACTION_NAMES,
                key=lambda a: Q[(state, a)]
            )
            print(f"{best:^8}", end=" ")
    print()


Training Completed
Episodes reaching goal: 1000

Best Action for Each State:
  DOWN     LEFT      UP       UP    
  DOWN      X        UP      DOWN   
 RIGHT     DOWN      X       DOWN   
   UP     RIGHT    RIGHT     GOAL   


In [ ]:
#TD - with epsilon
import random

ROWS, COLS = 4, 4
START = (0, 0)
GOAL = (3, 3)
RESTRICTED = [(1, 1), (2, 2)]

ALPHA = 0.1
EPSILON = 0.1
EPISODES = 1000
MAX_STEPS = 500

ACTIONS = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1)
}

V = {
    (r, c): 0.0
    for r in range(ROWS)
    for c in range(COLS)
    if (r, c) not in RESTRICTED
}

def move_agent(state, action):
    r, c = state
    dr, dc = ACTIONS[action]

    next_state = (r + dr, c + dc)

    if not (0 <= next_state[0] < ROWS and
            0 <= next_state[1] < COLS):
        return state

    if next_state in RESTRICTED:
        return state

    return next_state

# Epsilon-greedy using reward + V(next_state)
def choose_action(state):
    if random.random() < EPSILON:
        return random.choice(list(ACTIONS))

    action_values = {}

    for action in ACTIONS:
        next_state = move_agent(state, action)
        reward = 1 if next_state == GOAL else 0
        action_values[action] = reward + V[next_state]

    max_value = max(action_values.values())

    best_actions = [
        a for a in ACTIONS
        if action_values[a] == max_value
    ]

    return random.choice(best_actions)

# TD(0) training
successes = 0

for ep in range(EPISODES):
    state = START

    for step in range(MAX_STEPS):
        action = choose_action(state)
        next_state = move_agent(state, action)

        reward = 1 if next_state == GOAL else 0

        td_error = reward + V[next_state] - V[state]
        V[state] += ALPHA * td_error

        state = next_state

        if state == GOAL:
            successes += 1
            break

print("Training completed")
print("Episodes reaching goal:", successes)

print("\nFinal State Values:")
for r in range(ROWS):
    for c in range(COLS):
        s = (r, c)
        if s in RESTRICTED:
            print(f"{'X':>7}", end=" ")
        else:
            print(f"{V[s]:7.3f}", end=" ")
    print()

print("\nLearned Greedy Policy:")
for r in range(ROWS):
    for c in range(COLS):
        s = (r, c)

        if s == GOAL:
            print(f"{'GOAL':^8}", end=" ")
        elif s in RESTRICTED:
            print(f"{'X':^8}", end=" ")
        else:
            best = max(
                ACTIONS,
                key=lambda a: (
                    (1 if move_agent(s, a) == GOAL else 0)
                    + V[move_agent(s, a)]
                )
            )
            print(f"{best:^8}", end=" ")
    print()

Training completed
Episodes reaching goal: 1000

Final State Values:
  0.998   0.944   0.063   0.000 
  1.000       X   0.000   0.000 
  1.000   0.998       X   0.100 
  1.000   1.000   1.000   0.000 

Learned Greedy Policy:
  DOWN     LEFT     LEFT     LEFT   
  DOWN      X        UP      DOWN   
  DOWN     DOWN      X       DOWN   
 RIGHT    RIGHT    RIGHT     GOAL   


In [ ]:
#Dyna - Q
import numpy as np
import random

ACTIONS = ["NS Green", "EW Green", "NS Yellow", "EW Yellow"]

ALPHA = 0.1
GAMMA = 0.95
EPSILON = 0.1
EPISODES = 30
STEPS = 30

# Convert queue into low, medium, high
def traffic_level(q):
    if q <= 3:
        return 0
    if q <= 7:
        return 1
    return 2

def get_state(ns, ew, phase):
    return (traffic_level(ns), traffic_level(ew), phase)

# Epsilon-greedy selection
def choose_action(Q, state):
    if random.random() < EPSILON:
        return random.randrange(4)

    values = Q[state]
    best = np.flatnonzero(values == max(values))
    return int(random.choice(best))

# Q-learning update
def q_update(Q, s, a, r, next_s):
    target = r + GAMMA * max(Q[next_s])
    Q[s][a] += ALPHA * (target - Q[s][a])

# Traffic environment
def traffic_step(ns, ew, phase, action, rng):
    ns = min(20, ns + int(rng.poisson(1.5)))
    ew = min(20, ew + int(rng.poisson(1.5)))

    if action == 0:
        passed = min(ns, 4)
        ns -= passed
    elif action == 1:
        passed = min(ew, 4)
        ew -= passed
    elif action == 2:
        passed = min(ns, 1)
        ns -= passed
    else:
        passed = min(ew, 1)
        ew -= passed

    waiting = ns + ew
    congestion = max(ns, ew) / 20
    switching = int(action != phase)

    reward = (5 * passed - waiting
              - 0.2 * waiting
              - 5 * congestion
              - 2 * switching)

    return ns, ew, action, reward

# Dyna-Q training
def train(planning_steps):
    random.seed(42)
    rng = np.random.default_rng(42)

    Q = {}
    model = {}

    # Initialize all possible states
    for ns in range(3):
        for ew in range(3):
            for phase in range(4):
                Q[(ns, ew, phase)] = np.zeros(4)

    rewards = []

    for ep in range(EPISODES):
        ns = int(rng.integers(1, 6))
        ew = int(rng.integers(1, 6))
        phase = int(rng.integers(0, 4))

        total_reward = 0

        for step in range(STEPS):
            state = get_state(ns, ew, phase)
            action = choose_action(Q, state)

            ns, ew, phase, reward = traffic_step(
                ns, ew, phase, action, rng
            )

            next_state = get_state(ns, ew, phase)

            # Real Q-learning update
            q_update(Q, state, action, reward, next_state)

            # Store experience
            model[(state, action)] = (reward, next_state)

            # Planning updates
            for _ in range(planning_steps):
                s, a = random.choice(list(model.keys()))
                r, s_next = model[(s, a)]
                q_update(Q, s, a, r, s_next)

            total_reward += reward

        rewards.append(total_reward)

    return Q, rewards

# Compare different planning steps
for n in [0, 5, 10]:
    Q, rewards = train(n)

    print("Planning steps:", n)
    print("Average reward:", round(np.mean(rewards), 2))
    print("Average last 10:", round(np.mean(rewards[-10:]), 2))
    print()

# Demonstrate learned action for one state
Q, rewards = train(5)

state = (2, 0, 1)
best_action = np.argmax(Q[state])

print("Test state:", state)
print("Q-values:", np.round(Q[state], 2))
print("Best signal:", ACTIONS[best_action])

Planning steps: 0
Average reward: 197.09
Average last 10: 217.78

Planning steps: 5
Average reward: 162.32
Average last 10: 187.7

Planning steps: 10
Average reward: 203.0
Average last 10: 235.08

Test state: (2, 0, 1)
Q-values: [ 74.22  30.61 -18.09  46.07]
Best signal: NS Green


In [ ]:
#Reinforce
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

np.random.seed(7)
torch.manual_seed(7)

EPISODES = 200
GAMMA = 0.99
LR = 0.005

# Traffic Environment
class TrafficEnv:
    def reset(self):
        self.ns = np.random.randint(0, 4)
        self.ew = np.random.randint(0, 4)
        self.phase = 0
        self.steps = 0
        return self.state()

    def state(self):
        return np.array([
            self.ns / 20,
            self.ew / 20,
            float(self.phase)
        ], dtype=np.float32)

    def step(self, action):
        self.phase = int(action)

        self.ns = min(20, self.ns + np.random.poisson(1.4))
        self.ew = min(20, self.ew + np.random.poisson(1.4))

        service = np.random.randint(1, 4)

        if action == 0:
            self.ns = max(0, self.ns - service)
        else:
            self.ew = max(0, self.ew - service)

        reward = -(self.ns + self.ew)

        self.steps += 1
        done = self.steps >= 60

        return self.state(), float(reward), done


# Policy Network
class PolicyNetwork(nn.Module):
  def __init__(self):
    super().__init__()

    self.net = nn.Sequential(
        nn.Linear(3,32),
        nn.ReLU(),
        nn.Linear(32,32),
        nn.ReLU(),
        nn.Linear(32,2)
    )

  def forward(self, state):
      return self.net(state)

policy = PolicyNetwork()
optimizer = optim.Adam(policy.parameters(), lr = LR)

def discounted_returns(rewards):
  returns = []
  G = 0
  for r in reversed(rewards):
    G = r + GAMMA * G
    returns.insert(0,G)

  return torch.tensor(returns, dtype= torch.float32)

env = TrafficEnv()
reward_history = []
for ep in range(EPISODES):
  state = env.reset()
  log_probs = []
  rewards = []
  done = False

  while not done:
    s = torch.tensor(state).unsqueeze(0)
    #to get action probs - applies softmax
    dist = torch.distributions.Categorical(
        logits = policy(s)
    )
    action = dist.sample() #random selection
    log_probs.append(dist.log_prob(action).squeeze())

    state, reward, done = env.step(action.item())
    rewards.append(reward)


  G = discounted_returns(rewards)
  G = (G - G.mean()) / (G.std() + 1e-8) #normalization

  #policy gradient loss
  loss = -sum(lp*g for lp, g in zip(log_probs, G))
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  reward_history.append(sum(rewards))

print("Training completed")
print("Average reward (last 20 episodes):",
      round(np.mean(reward_history[-20:]), 2))


# Test learned policy
state = env.reset()
total_reward = 0

with torch.no_grad():
    for _ in range(60):
        s = torch.tensor(state).unsqueeze(0)
        probs = torch.softmax(policy(s), dim=1)

        action = torch.argmax(probs, dim=1).item()

        state, reward, done = env.step(action)
        total_reward += reward

        if done:
            break

print("Test episode reward:", total_reward)

# Test action probabilities for a sample state
sample_state = torch.tensor([[0.6, 0.2, 0.0]])

with torch.no_grad():
    probs = torch.softmax(policy(sample_state), dim=1)

print("NS Green probability:", round(probs[0, 0].item(), 3))
print("EW Green probability:", round(probs[0, 1].item(), 3))


Training completed
Average reward (last 20 episodes): -1261.3
Test episode reward: -1109.0
NS Green probability: 0.635
EW Green probability: 0.365


In [ ]:
#DQN
import random
import numpy as np
import torch
import torch.nn as nn
from collections import deque

# Simple vehicle environment
class Env:
    def reset(self):
        self.d = 25.0
        self.v = 12.0
        self.lead = 10.0
        self.steps = 0
        return self.state()

    def state(self):
        return np.array([self.d/40, self.v/30, self.lead/30],
                        dtype=np.float32)

    def step(self, action):
        acc = [2, 0, -4][action]
        self.v = np.clip(self.v + acc*0.2, 0, 30)
        self.d = max(0, self.d + (self.lead-self.v)*0.2)
        self.steps += 1

        collision = self.d < 2
        reward = -100 if collision else (
            0.08*self.v - 0.7*max(0, 12-self.d)
        )
        done = collision or self.steps >= 100
        return self.state(), reward, done

# Q-network
class DQN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 3)
        )

    def forward(self, x):
        return self.net(x)

online = DQN()
target = DQN()
target.load_state_dict(online.state_dict())

optimizer = torch.optim.Adam(online.parameters(), lr=0.001)
memory = deque(maxlen=20000)

GAMMA = 0.99
BATCH = 64
EPISODES = 100

env = Env()
total_steps = 0
rewards = []

for ep in range(EPISODES):
    state = env.reset()
    total_reward = 0
    done = False

    while not done:
        epsilon = 0.05 + 0.95*np.exp(-total_steps/7000)

        if random.random() < epsilon:
            action = random.randrange(3)
        else:
            with torch.no_grad():
                action = online(
                    torch.tensor(state).unsqueeze(0)
                ).argmax().item()

        next_state, reward, done = env.step(action)
        memory.append((state, action, reward, next_state, done))

        if len(memory) >= BATCH:
            batch = random.sample(memory, BATCH)
            s, a, r, ns, d = zip(*batch)

            s = torch.tensor(np.array(s), dtype=torch.float32)
            ns = torch.tensor(np.array(ns), dtype=torch.float32)
            a = torch.tensor(a).long().unsqueeze(1)
            r = torch.tensor(r, dtype=torch.float32)
            d = torch.tensor(d, dtype=torch.float32)

            current_q = online(s).gather(1, a).squeeze(1)

            with torch.no_grad():
                next_q = target(ns).max(1).values
                target_q = r + GAMMA*next_q*(1-d)

            loss = nn.SmoothL1Loss()(current_q, target_q)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        state = next_state
        total_reward += reward
        total_steps += 1

    rewards.append(total_reward)

    # Copy online weights to target every 10 episodes
    if (ep + 1) % 10 == 0:
        target.load_state_dict(online.state_dict())

print("Training completed")
print("Average reward (last 10):", np.mean(rewards[-10:]))

# Test learned policy
state = env.reset()
done = False
test_reward = 0

while not done:
    with torch.no_grad():
        action = online(
            torch.tensor(state).unsqueeze(0)
        ).argmax().item()

    state, reward, done = env.step(action)
    test_reward += reward

print("Test reward:", round(test_reward, 2))
print("Final distance:", round(env.d, 2))


Training completed
Average reward (last 10): 84.17319999999992
Test reward: 84.9
Final distance: 12.76


In [ ]:
#Actor-critic
import numpy as np

np.random.seed(7)

# Vehicle environment
class VehicleEnv:
    def reset(self):
        self.speed = np.random.randint(40, 66)
        self.distance = np.random.randint(18, 46)
        self.lead_speed = np.random.randint(45, 66)
        return self.state()

    def state(self):
        speed_bin = np.clip(int(self.speed // 10), 0, 9)
        distance_bin = np.clip(int(self.distance // 10), 0, 5)
        relative = self.lead_speed - self.speed
        relative_bin = 0 if relative < -5 else 2 if relative > 5 else 1
        return speed_bin, distance_bin, relative_bin

    def step(self, action):
        # 0 = Brake, 1 = Maintain, 2 = Accelerate
        self.speed += [-5, 0, 5][action]
        self.speed = np.clip(self.speed, 20, 90)

        self.lead_speed += np.random.choice([-3, 0, 0, 0, 3])
        self.lead_speed = np.clip(self.lead_speed, 35, 75)

        self.distance += 0.35 * (self.lead_speed - self.speed)
        self.distance = np.clip(self.distance, 5, 60)

        reward = 0

        if self.distance < 10:
            reward -= 20
            if action == 0:
                reward += 8
        elif self.distance < 20:
            reward -= 5
            if action == 0:
                reward += 4
        elif self.distance >= 30 and self.speed < 60:
            reward += 8 if action == 2 else -2
        elif 20 <= self.distance < 30:
            if abs(self.speed - 60) <= 5:
                reward += 5
            elif self.speed < 60 and action == 2:
                reward += 4
            else:
                reward += 1

        error = abs(self.speed - 60)
        reward += 8 if error == 0 else 4 if error <= 5 else -0.5

        return self.state(), reward


# Actor: action probabilities
policy = np.ones((10, 6, 3, 3)) / 3

# Critic: state values
V = np.zeros((10, 6, 3))

ALPHA_ACTOR = 0.2
ALPHA_CRITIC = 0.25
GAMMA = 0.8
EPISODES = 200
STEPS = 150

env = VehicleEnv()
rewards = []

# Training
for ep in range(EPISODES):
    state = env.reset()
    total_reward = 0

    for step in range(STEPS):
        # Actor selects action
        action = np.random.choice(3, p=policy[state])

        next_state, reward = env.step(action)

        # Critic calculates TD error
        td_error = reward + GAMMA * V[next_state] - V[state]

        # Update critic
        V[state] += ALPHA_CRITIC * td_error

        # Update actor
        policy[state][action] += ALPHA_ACTOR * td_error
        policy[state] = np.maximum(policy[state], 0.001)
        policy[state] /= policy[state].sum()

        total_reward += reward
        state = next_state

    rewards.append(total_reward)

print("Training completed")
print("Average reward (last 20 episodes):",
      round(np.mean(rewards[-20:]), 2))

# Test learned policy
state = env.reset()
actions = ["Brake", "Maintain", "Accelerate"]

print("\nTesting learned policy:")

for step in range(10):
    action = np.argmax(policy[state])
    state, reward = env.step(action)

    print(
        f"Speed: {env.speed:.1f} km/h | "
        f"Distance: {env.distance:.1f} m | "
        f"Action: {actions[action]} | "
        f"Reward: {reward:.1f}"
    )


Training completed
Average reward (last 20 episodes): 509.55

Testing learned policy:
Speed: 48.0 km/h | Distance: 44.6 m | Action: Accelerate | Reward: 7.5
Speed: 53.0 km/h | Distance: 43.6 m | Action: Accelerate | Reward: 7.5
Speed: 58.0 km/h | Distance: 40.8 m | Action: Accelerate | Reward: 12.0
Speed: 63.0 km/h | Distance: 35.2 m | Action: Accelerate | Reward: 4.0
Speed: 58.0 km/h | Distance: 31.4 m | Action: Brake | Reward: 2.0
Speed: 58.0 km/h | Distance: 27.5 m | Action: Maintain | Reward: 9.0
Speed: 53.0 km/h | Distance: 25.4 m | Action: Brake | Reward: 0.5
Speed: 48.0 km/h | Distance: 24.0 m | Action: Brake | Reward: 0.5
Speed: 43.0 km/h | Distance: 24.4 m | Action: Brake | Reward: 0.5
Speed: 38.0 km/h | Distance: 25.4 m | Action: Brake | Reward: 0.5
